# Sistema Multiagente: Retención de Marca + Optimización de Infraestructura
### Orquestador + 2 Subagentes sobre arquitectura Medallion (OneLake)

Este notebook implementa un **sistema de agentes** compuesto por:

- **Agente Orquestador**: coordina la ejecución de tareas entre subagentes, gestiona prioridades y registra auditoría.
- **Subagente 1 — Analista de Negocio y Retención**: análisis de salud de marca, eliminación de silos de datos (CRM, marketing, uso) y modelo de riesgo de abandono (churn).
- **Subagente 2 — Optimizador de Infraestructura y Costos**: monitoreo de capacidad, autoescalado de registros/inscripciones y automatización de pausas de F-SKU (capacidades tipo Microsoft Fabric).

Todo el flujo de datos sigue una **arquitectura Medallion (Bronze → Silver → Gold)**, simulando una implementación sobre **OneLake**:

| Capa | Contenido | Propósito |
|---|---|---|
| **Bronze** | Datos crudos de cada silo (CRM, marketing, uso, infraestructura, catálogo F-SKU) | Ingesta sin transformar, trazabilidad total |
| **Silver** | `customer_360` — silos unificados | Fuente única de verdad, sin duplicidad ni fragmentación |
| **Gold** | KPIs de marca, scores de riesgo, decisiones de escalado, plan de pausa F-SKU, log de auditoría | Consumo directo por negocio / dashboards |

> **Nota de entorno:** este notebook usa datos sintéticos y persiste las capas como CSV (en vez de Delta/Parquet) para poder ejecutarse sin dependencias externas ni acceso a red. La lógica y contratos de datos son directamente portables a OneLake/Microsoft Fabric (Lakehouse + Delta tables) o a cualquier data lake (S3 + Delta/Iceberg).

## 1. Configuración e imports

In [ ]:
import os, json, uuid, random, datetime as dt
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score

random.seed(42)
np.random.seed(42)
pd.set_option("display.max_columns", 50)
plt.rcParams["figure.figsize"] = (9, 4)


## 2. Simulación de OneLake — Arquitectura Medallion

Se crea la estructura de carpetas `bronze/`, `silver/`, `gold/` y funciones de lectura/escritura genéricas (`lake_read` / `lake_write`) que abstraen el motor de almacenamiento.

En una implementación real sobre **Microsoft Fabric / OneLake**, `lake_write` escribiría tablas Delta dentro de un Lakehouse (`abfss://...@onelake.dfs.fabric.microsoft.com/...`), y `lake_read` usaría Spark o el conector de OneLake. La interfaz (`layer`, `name`, `DataFrame`) se mantiene idéntica, por lo que migrar de esta simulación a producción es un cambio de implementación, no de arquitectura.

In [ ]:
LAKE_ROOT = "./onelake_sim"
LAYERS = {
    "bronze": os.path.join(LAKE_ROOT, "bronze"),   # datos crudos, tal cual llegan de cada silo
    "silver": os.path.join(LAKE_ROOT, "silver"),   # datos limpios y unificados (sin silos)
    "gold":   os.path.join(LAKE_ROOT, "gold"),     # datos listos para negocio / KPIs / modelos
}
for p in LAYERS.values():
    os.makedirs(p, exist_ok=True)

def lake_write(layer: str, name: str, df: pd.DataFrame) -> str:
    """Persiste un DataFrame en la capa indicada del lake (CSV local == Delta/Parquet en OneLake)."""
    path = os.path.join(LAYERS[layer], f"{name}.csv")
    df.to_csv(path, index=False)
    return path

def lake_read(layer: str, name: str) -> pd.DataFrame:
    """Lee una tabla desde la capa indicada del lake."""
    path = os.path.join(LAYERS[layer], f"{name}.csv")
    return pd.read_csv(path)

print("Medallion (OneLake simulado) inicializado en:", os.path.abspath(LAKE_ROOT))


## 3. Datos sintéticos — fuentes "siloadas"

Se generan 6 datasets que representan los silos típicos de una organización:

- `crm_silo`: plan, antigüedad, MRR, tickets de soporte
- `marketing_silo`: menciones de marca, sentimiento, NPS
- `usage_silo`: logins, adopción de features, inactividad
- `churn_labels`: etiqueta histórica de abandono (para entrenar el modelo)
- `infra_metrics`: utilización de capacidad, registros activos, profundidad de cola (168h = 1 semana)
- `fsku_catalog`: catálogo de F-SKUs (capacidades tipo Microsoft Fabric) con costo/hora

Estos datasets se ingieren tal cual en **Bronze**, sin ninguna transformación.

In [ ]:
N_CUSTOMERS = 800

def gen_crm_silo():
    return pd.DataFrame({
        "customer_id": [f"C{100000+i}" for i in range(N_CUSTOMERS)],
        "plan": np.random.choice(["Free", "Pro", "Enterprise"], N_CUSTOMERS, p=[.5, .35, .15]),
        "tenure_months": np.random.randint(1, 48, N_CUSTOMERS),
        "mrr": np.round(np.random.gamma(2, 40, N_CUSTOMERS), 2),
        "support_tickets_90d": np.random.poisson(1.2, N_CUSTOMERS),
    })

def gen_marketing_silo():
    return pd.DataFrame({
        "customer_id": [f"C{100000+i}" for i in range(N_CUSTOMERS)],
        "brand_mentions_90d": np.random.poisson(0.8, N_CUSTOMERS),
        "sentiment_score": np.round(np.random.normal(0.15, 0.4, N_CUSTOMERS), 2).clip(-1, 1),
        "nps": np.random.randint(-100, 100, N_CUSTOMERS),
    })

def gen_usage_silo():
    return pd.DataFrame({
        "customer_id": [f"C{100000+i}" for i in range(N_CUSTOMERS)],
        "logins_30d": np.random.poisson(6, N_CUSTOMERS),
        "feature_adoption_pct": np.round(np.random.beta(2, 2, N_CUSTOMERS) * 100, 1),
        "days_since_last_login": np.random.exponential(10, N_CUSTOMERS).astype(int),
    })

def gen_churn_labels(crm, mkt, usage):
    # Etiqueta sintética de abandono, influenciada por señales realistas de negocio
    risk = (
        0.03 * crm["support_tickets_90d"]
        - 0.01 * crm["tenure_months"]
        - 0.6 * mkt["sentiment_score"]
        - 0.02 * usage["logins_30d"]
        + 0.03 * usage["days_since_last_login"]
        - 0.01 * usage["feature_adoption_pct"]
        + np.random.normal(0, 1, N_CUSTOMERS)
    )
    prob = 1 / (1 + np.exp(-(risk - 2)))
    churn = (np.random.rand(N_CUSTOMERS) < prob).astype(int)
    return pd.DataFrame({"customer_id": crm["customer_id"], "churned": churn})

def gen_infra_metrics(hours=24 * 7):
    ts = pd.date_range(end=dt.datetime.now(dt.timezone.utc), periods=hours, freq="h")
    base = 40 + 25 * np.sin(np.linspace(0, 6 * np.pi, hours)) + np.random.normal(0, 5, hours)
    return pd.DataFrame({
        "timestamp": ts,
        "capacity_utilization_pct": base.clip(5, 100).round(1),
        "active_registrations": np.random.poisson(120, hours) + (base > 70) * 80,
        "queue_depth": np.random.poisson(3, hours),
    })

def gen_fsku_catalog():
    skus = [f"F{2**i}" for i in range(5)]  # F1, F2, F4, F8, F16 (estilo Microsoft Fabric)
    return pd.DataFrame({
        "sku": skus,
        "cost_per_hour_usd": [0.36, 0.72, 1.44, 2.88, 5.76],
        "capacity_units": [1, 2, 4, 8, 16],
        "status": ["active"] * len(skus),
    })

crm_raw, mkt_raw, usage_raw = gen_crm_silo(), gen_marketing_silo(), gen_usage_silo()
churn_raw = gen_churn_labels(crm_raw, mkt_raw, usage_raw)
infra_raw = gen_infra_metrics()
fsku_raw = gen_fsku_catalog()

for name, df in [("crm_silo", crm_raw), ("marketing_silo", mkt_raw),
                  ("usage_silo", usage_raw), ("churn_labels", churn_raw),
                  ("infra_metrics", infra_raw), ("fsku_catalog", fsku_raw)]:
    lake_write("bronze", name, df)

print("Bronze: 6 datasets crudos ingeridos.")
lake_read("bronze", "crm_silo").head()


## 4. Agente Orquestador

Responsabilidades:

- Recibir tareas (`submit_task`) con tipo, subagente destino, parámetros y prioridad — equivalente al endpoint `POST /api/orchestrator/tasks` de la arquitectura de referencia.
- Ejecutarlas en orden (`run`), invocando dinámicamente el método correspondiente del subagente.
- Registrar un **log de auditoría** (`audit_log_df`) con tiempos de ejecución y estado — persistido en Gold para trazabilidad, gobierno y cumplimiento.

In [ ]:
class OrchestratorAgent:
    """Coordina subagentes, gestiona una cola de tareas y registra auditoría en Gold."""

    def __init__(self):
        self.tasks = []
        self.logs = []

    def submit_task(self, task_type: str, agent_name: str, params: dict, priority: str = "medium") -> str:
        task = {
            "task_id": str(uuid.uuid4())[:8],
            "task_type": task_type,
            "agent": agent_name,
            "params": params,
            "priority": priority,
            "status": "queued",
            "created_at": dt.datetime.now(dt.timezone.utc).isoformat(),
        }
        self.tasks.append(task)
        return task["task_id"]

    def run(self, agents: dict) -> dict:
        results = {}
        for task in self.tasks:
            agent = agents[task["agent"]]
            fn = getattr(agent, task["task_type"])
            t0 = dt.datetime.now(dt.timezone.utc)
            output = fn(**task["params"])
            task["status"] = "completed"
            elapsed = (dt.datetime.now(dt.timezone.utc) - t0).total_seconds()
            self.logs.append({
                "task_id": task["task_id"], "agent": task["agent"], "task_type": task["task_type"],
                "priority": task["priority"], "status": "completed",
                "elapsed_sec": elapsed, "timestamp": dt.datetime.now(dt.timezone.utc).isoformat(),
            })
            results[task["task_id"]] = output
        return results

    def audit_log_df(self) -> pd.DataFrame:
        return pd.DataFrame(self.logs)


## 5. Subagente 1 — Analista de Negocio y Retención

| Método | Función | Salida en Gold |
|---|---|---|
| `eliminate_silos()` | Une CRM + marketing + uso + churn en una vista `customer_360` | `silver/customer_360` |
| `analyze_brand()` | Calcula salud de marca: sentimiento, NPS, menciones | `gold/brand_health` |
| `churn_risk_model()` | Entrena un `RandomForestClassifier` y genera score de riesgo por cliente | `gold/churn_risk_scores` |

In [ ]:
class BusinessRetentionAgent:
    """Análisis de marca, eliminación de silos y modelado de riesgo de abandono."""

    def eliminate_silos(self) -> dict:
        crm = lake_read("bronze", "crm_silo")
        mkt = lake_read("bronze", "marketing_silo")
        usage = lake_read("bronze", "usage_silo")
        churn = lake_read("bronze", "churn_labels")
        unified = (crm.merge(mkt, on="customer_id")
                       .merge(usage, on="customer_id")
                       .merge(churn, on="customer_id"))
        lake_write("silver", "customer_360", unified)
        return {"rows": len(unified), "columns": list(unified.columns), "layer": "silver/customer_360"}

    def analyze_brand(self) -> dict:
        mkt = lake_read("bronze", "marketing_silo")
        summary = {
            "sentiment_promedio": round(float(mkt["sentiment_score"].mean()), 3),
            "nps_promedio": round(float(mkt["nps"].mean()), 1),
            "menciones_90d_totales": int(mkt["brand_mentions_90d"].sum()),
            "pct_sentimiento_negativo": round(float((mkt["sentiment_score"] < 0).mean() * 100), 1),
        }
        lake_write("gold", "brand_health", pd.DataFrame([summary]))
        return summary

    def churn_risk_model(self) -> dict:
        df = lake_read("silver", "customer_360")
        features = ["tenure_months", "mrr", "support_tickets_90d", "brand_mentions_90d",
                    "sentiment_score", "nps", "logins_30d", "feature_adoption_pct", "days_since_last_login"]
        X, y = df[features], df["churned"]
        X_train, X_test, y_train, y_test = train_test_split(
            X, y, test_size=0.25, random_state=42, stratify=y
        )
        model = RandomForestClassifier(n_estimators=200, max_depth=6, random_state=42, class_weight="balanced")
        model.fit(X_train, y_train)
        auc = roc_auc_score(y_test, model.predict_proba(X_test)[:, 1])

        df["risk_score"] = model.predict_proba(X)[:, 1]
        df["risk_tier"] = pd.cut(df["risk_score"], [0, 0.33, 0.66, 1.0], labels=["bajo", "medio", "alto"])
        scored = df[["customer_id", "plan", "mrr", "risk_score", "risk_tier"]].sort_values("risk_score", ascending=False)
        lake_write("gold", "churn_risk_scores", scored)

        importances = pd.Series(model.feature_importances_, index=features).sort_values(ascending=False)
        return {
            "auc_test": round(float(auc), 3),
            "clientes_riesgo_alto": int((df["risk_tier"] == "alto").sum()),
            "mrr_en_riesgo_alto": round(float(df.loc[df["risk_tier"] == "alto", "mrr"].sum()), 2),
            "top_features": importances.head(4).round(3).to_dict(),
        }


## 6. Subagente 2 — Optimizador de Infraestructura y Costos

| Método | Función | Salida en Gold |
|---|---|---|
| `monitor_capacity()` | KPIs de utilización de capacidad y profundidad de cola | `gold/capacity_summary` |
| `autoscale_registrations()` | Decide escalar/reducir nodos según utilización y cola, por hora | `gold/autoscaling_decisions` |
| `automate_fsku_pause()` | Recomienda pausar F-SKUs grandes si el % de horas ociosas supera el umbral | `gold/fsku_pause_plan` |

Los umbrales (`CAP_HIGH`, `CAP_LOW`, umbral de horas ociosas) son parámetros de negocio: se recomienda gobernarlos desde una tabla de configuración en Gold en vez de hardcodearlos en producción.

In [ ]:
class InfraCostAgent:
    """Monitoreo de capacidad, autoescalado de registros/inscripciones y pausa automática de F-SKUs."""

    CAP_HIGH, CAP_LOW = 75.0, 20.0   # umbrales % de utilización de capacidad
    IDLE_RATIO_THRESHOLD = 0.15      # % de horas ociosas que dispara recomendación de pausa

    def monitor_capacity(self) -> dict:
        infra = lake_read("bronze", "infra_metrics")
        summary = {
            "utilizacion_promedio_pct": round(float(infra["capacity_utilization_pct"].mean()), 1),
            "utilizacion_max_pct": round(float(infra["capacity_utilization_pct"].max()), 1),
            "horas_sobre_umbral_alto": int((infra["capacity_utilization_pct"] > self.CAP_HIGH).sum()),
            "horas_bajo_umbral_bajo": int((infra["capacity_utilization_pct"] < self.CAP_LOW).sum()),
            "queue_depth_promedio": round(float(infra["queue_depth"].mean()), 2),
        }
        lake_write("gold", "capacity_summary", pd.DataFrame([summary]))
        return summary

    def autoscale_registrations(self) -> dict:
        infra = lake_read("bronze", "infra_metrics").copy()

        def decide(row):
            if row["capacity_utilization_pct"] > self.CAP_HIGH or row["queue_depth"] > 6:
                return "escalar_+1_nodo"
            if row["capacity_utilization_pct"] < self.CAP_LOW and row["queue_depth"] == 0:
                return "reducir_-1_nodo"
            return "mantener"

        infra["scaling_action"] = infra.apply(decide, axis=1)
        lake_write("gold", "autoscaling_decisions",
                   infra[["timestamp", "capacity_utilization_pct", "queue_depth", "scaling_action"]])
        return {"acciones_periodo": infra["scaling_action"].value_counts().to_dict(), "horas_evaluadas": len(infra)}

    def automate_fsku_pause(self) -> dict:
        infra = lake_read("bronze", "infra_metrics")
        fsku = lake_read("bronze", "fsku_catalog").copy()
        low_usage_hours = int((infra["capacity_utilization_pct"] < self.CAP_LOW).sum())
        idle_ratio = low_usage_hours / len(infra)

        fsku["recommended_status"] = np.where(
            (idle_ratio > self.IDLE_RATIO_THRESHOLD) & (fsku["capacity_units"] >= 8), "pausar", "activo"
        )
        costo_ahorrado = float(
            fsku.loc[fsku["recommended_status"] == "pausar", "cost_per_hour_usd"].sum()
        ) * low_usage_hours
        lake_write("gold", "fsku_pause_plan", fsku)
        return {
            "pct_horas_ociosas": round(idle_ratio * 100, 1),
            "skus_a_pausar": fsku.loc[fsku["recommended_status"] == "pausar", "sku"].tolist(),
            "ahorro_estimado_usd_periodo": round(costo_ahorrado, 2),
        }


## 7. Ejecución orquestada end-to-end

El orquestador encola las 6 tareas (3 por subagente) respetando dependencias implícitas (`eliminate_silos` antes que `churn_risk_model`, ya que este último lee de `silver/customer_360`) y las ejecuta secuencialmente. En una versión productiva con LangGraph/CrewAI esto se modelaría como un grafo de dependencias explícito con paralelismo entre subagentes independientes.

In [ ]:
orchestrator = OrchestratorAgent()
retention_agent = BusinessRetentionAgent()
infra_agent = InfraCostAgent()
agents = {"retention": retention_agent, "infra": infra_agent}

orchestrator.submit_task("eliminate_silos", "retention", {}, priority="high")
orchestrator.submit_task("analyze_brand", "retention", {}, priority="medium")
orchestrator.submit_task("churn_risk_model", "retention", {}, priority="high")
orchestrator.submit_task("monitor_capacity", "infra", {}, priority="high")
orchestrator.submit_task("autoscale_registrations", "infra", {}, priority="medium")
orchestrator.submit_task("automate_fsku_pause", "infra", {}, priority="medium")

results = orchestrator.run(agents)
audit = orchestrator.audit_log_df()
lake_write("gold", "orchestrator_audit_log", audit)

print(json.dumps(results, indent=2, ensure_ascii=False))


In [ ]:
audit

## 8. Dashboard — Capa Gold

Visualización directa de los artefactos generados en Gold: salud de marca, distribución de riesgo de abandono, decisiones de autoescalado y utilización de capacidad.

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(13, 8))

# 1) Distribución de riesgo de abandono
risk = lake_read("gold", "churn_risk_scores")
risk["risk_tier"].value_counts().reindex(["bajo","medio","alto"]).plot(
    kind="bar", ax=axes[0,0], color=["#3fb950","#d29922","#f85149"]
)
axes[0,0].set_title("Clientes por nivel de riesgo de abandono")
axes[0,0].set_xlabel("")
axes[0,0].set_ylabel("# clientes")

# 2) Utilización de capacidad en el tiempo
infra = lake_read("bronze", "infra_metrics")
infra["timestamp"] = pd.to_datetime(infra["timestamp"])
axes[0,1].plot(infra["timestamp"], infra["capacity_utilization_pct"], color="#58a6ff")
axes[0,1].axhline(InfraCostAgent.CAP_HIGH, color="#f85149", linestyle="--", linewidth=1, label="umbral alto")
axes[0,1].axhline(InfraCostAgent.CAP_LOW, color="#3fb950", linestyle="--", linewidth=1, label="umbral bajo")
axes[0,1].set_title("Utilización de capacidad (última semana)")
axes[0,1].set_ylabel("% utilización")
axes[0,1].legend(fontsize=8)

# 3) Acciones de autoescalado
scaling = lake_read("gold", "autoscaling_decisions")
scaling["scaling_action"].value_counts().plot(kind="barh", ax=axes[1,0], color="#8957e5")
axes[1,0].set_title("Decisiones de autoescalado (168h)")
axes[1,0].set_xlabel("# horas")

# 4) MRR en riesgo por tier
mrr_por_tier = risk.groupby("risk_tier")["mrr"].sum().reindex(["bajo","medio","alto"])
mrr_por_tier.plot(kind="bar", ax=axes[1,1], color=["#3fb950","#d29922","#f85149"])
axes[1,1].set_title("MRR total en riesgo por nivel")
axes[1,1].set_ylabel("MRR (USD)")
axes[1,1].set_xlabel("")

plt.tight_layout()
plt.show()


## 9. Resumen ejecutivo (Gold → negocio)

In [ ]:
brand = lake_read("gold", "brand_health").iloc[0]
cap = lake_read("gold", "capacity_summary").iloc[0]
fsku_plan = lake_read("gold", "fsku_pause_plan")

print("== Salud de marca ==")
print(brand.to_string())
print()
print("== Capacidad de infraestructura ==")
print(cap.to_string())
print()
print("== Plan de F-SKU ==")
print(fsku_plan.to_string(index=False))


## 10. Próximos pasos hacia producción

1. **Persistencia real en OneLake**: sustituir `lake_write`/`lake_read` por escritura/lectura de tablas Delta en un Lakehouse de Microsoft Fabric (o Delta Lake sobre S3/ADLS si se prefiere multicloud).
2. **Orquestación como grafo**: migrar el `OrchestratorAgent` a **LangGraph** o **CrewAI** para manejar dependencias, reintentos, paralelismo y checkpoints de forma nativa.
3. **Conexión a fuentes reales**: reemplazar los generadores sintéticos por conectores reales (CRM, plataforma de marca/social listening, telemetría de infraestructura, API de Fabric Capacity para pausar/reanudar F-SKUs).
4. **Gobierno**: añadir el filtro de seguridad/aprobación (`Safety/Approval Filter`) antes de ejecutar acciones automáticas de alto impacto (pausar SKUs, escalar infraestructura).
5. **Observabilidad**: exponer `orchestrator_audit_log` en un dashboard operativo (Power BI directamente sobre OneLake, o Grafana) para practicar la operabilidad del sistema en tiempo real — ver el archivo visual adjunto (`orquestador_dashboard.html`) para una demo interactiva de este flujo.